In [1]:
from __future__ import annotations

import json
import os
import glob
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

# Ensure non-interactive backend (no display)
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from pynwb import NWBHDF5IO  # noqa: E402
from aind_dynamic_foraging_models.generative_model import ForagerCollection  # noqa: E402


# -----------------------------
# User config
# -----------------------------
LOCAL_NWB_TMP = "/data/foraging_nwb_bonsai"
RESULTS_ROOT = Path("/root/capsule/scratch/model_comparison")
MIN_VALID_TRIALS = 10

# Differential evolution config
DE_KWARGS = dict(
    workers=4,
    disp=False,
    seed=np.random.default_rng(42),
)

# If True, save fitted-session plots (still not displayed)
SAVE_FIGS = False
FIG_DPI = 150

# Parallelism across sessions
MAX_WORKERS = max(1, (os.cpu_count() or 2) // 2)


# -----------------------------
# NWB helpers
# -----------------------------
def get_nwb_from_local_tmp(session_id: str):
    """Get NWB file from session_id in LOCAL_NWB_TMP."""
    io = NWBHDF5IO(f"{LOCAL_NWB_TMP}/{session_id}.nwb", mode="r")
    nwb = io.read()
    return io, nwb


def get_history_from_nwb(nwb) -> Tuple[bool, np.ndarray, np.ndarray, np.ndarray]:
    """
    Get choice and reward history from NWB file.

    Returns
    -------
    baiting : bool
    choice_history : np.ndarray of shape (n_trials,), values in {0,1} (NaNs removed later)
    reward_history : np.ndarray of shape (n_trials,), dtype bool/int
    autowater_offered : np.ndarray of shape (n_trials,), dtype bool
    """
    df_trial = nwb.trials.to_dataframe()

    autowater_offered = (df_trial.auto_waterL == 1) | (df_trial.auto_waterR == 1)
    choice_history = df_trial.animal_response.map({0: 0, 1: 1, 2: np.nan}).values

    # Keep reward_history consistent with your original logic:
    reward_history = (df_trial.rewarded_historyL | df_trial.rewarded_historyR).to_numpy()

    baiting = False if "without baiting" in (nwb.protocol or "").lower() else True

    return baiting, choice_history, reward_history, autowater_offered.to_numpy()


# -----------------------------
# Forager specs (edit this list to add models)
# -----------------------------
@dataclass(frozen=True)
class ForagerSpec:
    """Specification for a forager model to fit."""

    name: str
    preset: Optional[str] = None
    agent_class_name: Optional[str] = None
    agent_kwargs: Optional[Dict[str, Any]] = None
    clamp_params: Optional[Dict[str, Any]] = None


def build_forager(spec: ForagerSpec):
    """Instantiate a forager from a preset or class_name + kwargs."""
    fc = ForagerCollection()
    if spec.preset is not None:
        return fc.get_preset_forager(spec.preset)
    if spec.agent_class_name is not None:
        return fc.get_forager(
            agent_class_name=spec.agent_class_name,
            agent_kwargs=spec.agent_kwargs or {},
        )
    raise ValueError(f"Invalid ForagerSpec: {spec}")


# Adjusted agent kwargs to match expected parameters for ForagerCompareThreshold
FORAGERS: List[ForagerSpec] = [
    ForagerSpec(
        name="ForagingCompareThreshold_L1",
        agent_class_name="ForagerCompareThreshold",
        agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": True},
    ),
    ForagerSpec(
        name="ForagingCompareThreshold_L1_NoReset",
        agent_class_name="ForagerCompareThreshold",
        agent_kwargs={"number_of_learning_rate": 1, "reset_to_threshold": False},
    ),
    ForagerSpec(
        name="ForagingCompareThreshold_L2",
        agent_class_name="ForagerCompareThreshold",
        agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": True},
    ),
    ForagerSpec(
        name="ForagingCompareThreshold_L2_NoReset",
        agent_class_name="ForagerCompareThreshold",
        agent_kwargs={"number_of_learning_rate": 2, "reset_to_threshold": False},
    ),
    # Adding the preset models Hattori2019 and Bari2019
    ForagerSpec(name="Hattori2019", preset="Hattori2019"),
    ForagerSpec(name="Bari2019", preset="Bari2019"),
]


# -----------------------------
# Serialization helpers
# -----------------------------
def _to_jsonable(x: Any) -> Any:
    """Best-effort conversion to JSON-serializable types."""
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if isinstance(x, (list, tuple)):
        return [_to_jsonable(v) for v in x]
    if isinstance(x, dict):
        return {str(k): _to_jsonable(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.integer, np.floating, np.bool_)):
        return x.item()
    return str(x)


def save_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(_to_jsonable(payload), f, indent=2)


# -----------------------------
# Core fit routine (per session)
# -----------------------------
def fit_one_session(session_path: str) -> Dict[str, Any]:
    """
    Fit all FORAGERS for a single NWB session.
    Saves outputs under RESULTS_ROOT/<session_id>/.
    Returns a lightweight summary dict for aggregation.
    """
    session_id = Path(session_path).stem
    out_dir = RESULTS_ROOT / session_id
    out_dir.mkdir(parents=True, exist_ok=True)

    session_summary: Dict[str, Any] = {
        "session_id": session_id,
        "nwb_path": session_path,
        "status": "ok",
        "n_valid_trials": None,
        "models": {},
    }

    io = None
    try:
        io, nwb = get_nwb_from_local_tmp(session_id=session_id)
        baiting, choice_history, reward_history, autowater_offered = get_history_from_nwb(nwb)

        # ----------------------------------------
        # Extract auto_train_stage (if exists)
        # ----------------------------------------
        auto_train_stage_last = None
        try:
            if hasattr(nwb, "trials") and "auto_train_stage" in nwb.trials.colnames:
                col = nwb.trials["auto_train_stage"].data[:]
                if len(col) > 0:
                    auto_train_stage_last = col[-1]
        except Exception:
            auto_train_stage_last = None

        session_summary["auto_train_stage"] = (
            None if auto_train_stage_last is None else _to_jsonable(auto_train_stage_last)
        )

        # Remove NaN choices
        ignored = np.isnan(choice_history)
        choice_valid = choice_history[~ignored].astype(int)
        reward_valid = reward_history[~ignored].astype(int)

        session_summary["n_valid_trials"] = int(len(choice_valid))
        session_summary["baiting"] = bool(baiting)

        if len(choice_valid) < MIN_VALID_TRIALS:
            session_summary["status"] = f"skipped: valid trials < {MIN_VALID_TRIALS}"
            save_json(out_dir / "summary.json", session_summary)
            return session_summary

        # Fit each model
        for spec in FORAGERS:
            model_key = spec.name
            model_out: Dict[str, Any] = {"status": "ok"}

            # NEW: add stage info into each model fit-results JSON
            model_out["auto_train_stage"] = session_summary.get("auto_train_stage", None)

            try:
                forager = build_forager(spec)

                fitting_result, _ = forager.fit(
                    choice_valid,
                    reward_valid,
                    clamp_params=spec.clamp_params or {},
                    DE_kwargs=DE_KWARGS,
                    # k_fold_cross_validation=None
                )

                # Create a dedicated folder for each model fitting
                model_folder = out_dir / model_key
                model_folder.mkdir(parents=True, exist_ok=True)

                # Save fitting result using pickle
                try:
                    import pickle

                    with (model_folder / "fitting_result.pkl").open("wb") as f:
                        pickle.dump(fitting_result, f)
                    model_out["pickle_saved"] = True
                except Exception as e:
                    model_out["pickle_saved"] = False
                    model_out["pickle_error"] = f"{type(e).__name__}: {e}"

                # Pull key numbers safely
                fit_settings = getattr(fitting_result, "fit_settings", {}) or {}
                fit_names = fit_settings.get("fit_names", None)

                model_out.update(
                    dict(
                        n_trials=int(len(choice_valid)),
                        LPT=float(getattr(fitting_result, "LPT", np.nan)),
                        AIC=float(getattr(fitting_result, "AIC", np.nan)),
                        BIC=float(getattr(fitting_result, "BIC", np.nan)),
                        prediction_accuracy=float(
                            getattr(fitting_result, "prediction_accuracy", np.nan)
                        ),
                        fit_names=fit_names,
                        x=[float(v) for v in list(getattr(fitting_result, "x", []))],
                    )
                )

                # Save a JSON version (includes auto_train_stage now)
                save_json(model_folder / f"{model_key}.json", model_out)

                # Optional figure saving (no display)
                if SAVE_FIGS:
                    try:
                        fig, _axes = forager.plot_fitted_session(if_plot_latent=True)
                        fig.savefig(model_folder / f"{model_key}.png", dpi=FIG_DPI)
                        plt.close(fig)
                        model_out["fig_saved"] = True
                    except Exception as e:
                        model_out["fig_saved"] = False
                        model_out["fig_error"] = f"{type(e).__name__}: {e}"

            except Exception as e:
                model_out["status"] = "error"
                model_out["error"] = f"{type(e).__name__}: {e}"
                model_out["traceback"] = traceback.format_exc()

                # Still write JSON so you can see failures per model
                save_json(out_dir / f"{model_key}.json", model_out)

            session_summary["models"][model_key] = model_out

        # Save per-session rollup
        save_json(out_dir / "summary.json", session_summary)
        return session_summary

    except Exception as e:
        session_summary["status"] = "error"
        session_summary["error"] = f"{type(e).__name__}: {e}"
        session_summary["traceback"] = traceback.format_exc()
        save_json(out_dir / "summary.json", session_summary)
        return session_summary

    finally:
        try:
            if io is not None:
                io.close()
        except Exception:
            pass


# -----------------------------
# Batch runner (parallel by session)
# -----------------------------
def find_all_sessions(local_root: str) -> List[str]:
    """Return all .nwb file paths under local_root."""
    return sorted(glob.glob(f"{local_root}/*.nwb"))


def main() -> None:
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

    session_paths = find_all_sessions(LOCAL_NWB_TMP)
    if len(session_paths) == 0:
        raise RuntimeError(f"No .nwb files found under: {LOCAL_NWB_TMP}")

    # Parallel across sessions
    from concurrent.futures import ProcessPoolExecutor, as_completed

    all_summaries: List[Dict[str, Any]] = []
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(fit_one_session, p): p for p in session_paths}
        for fut in as_completed(futures):
            all_summaries.append(fut.result())

    # Save global summary (one file)
    global_summary = {
        "local_root": LOCAL_NWB_TMP,
        "results_root": str(RESULTS_ROOT),
        "n_sessions_found": int(len(session_paths)),
        "n_sessions_processed": int(len(all_summaries)),
        "foragers": [spec.__dict__ for spec in FORAGERS],
        "summaries": all_summaries,
    }
    save_json(RESULTS_ROOT / "ALL_SESSIONS_SUMMARY.json", global_summary)


if __name__ == "__main__":
    main()

/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.4.0, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)
/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.4.0, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)
/opt/conda/lib/python3.10/site-packages/hdmf/spec/namespace.py:620: UserWarnin

2026-02-21 00:34:08,262 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...
2026-02-21 00:34:09,685 - aind_dynamic_foraging_models.generative_model.base - INFO - Fitting the model using the whole dataset...


In [1]:
import pickle

file_path = "/root/capsule/scratch/model_comparison/662914_2023-09-29/ForagingCompareThreshold_L1/fitting_result.pkl"

with open(file_path, "rb") as f:
    result = pickle.load(f)

print(type(result))

if isinstance(result, dict):
    print("Keys:", result.keys())

    # Print top-level summary
    for k, v in result.items():
        if hasattr(v, "__shape__"):
            print(k, "shape:", getattr(v, "shape", None))
        else:
            print(k, "type:", type(v))

<class 'scipy.optimize._optimize.OptimizeResult'>
Keys: dict_keys(['x', 'fun', 'nfev', 'nit', 'message', 'success', 'population', 'population_energies', 'jac', 'fit_settings', 'params', 'k_model', 'n_trials', 'log_likelihood', 'AIC', 'BIC', 'LPT', 'LPT_AIC', 'LPT_BIC', 'x_without_polishing', 'log_likelihood_without_polishing', 'params_without_polishing', 'prediction_accuracy'])
x type: <class 'numpy.ndarray'>
fun type: <class 'numpy.float64'>
nfev type: <class 'int'>
nit type: <class 'int'>
message type: <class 'str'>
success type: <class 'bool'>
population type: <class 'numpy.ndarray'>
population_energies type: <class 'numpy.ndarray'>
jac type: <class 'numpy.ndarray'>
fit_settings type: <class 'dict'>
params type: <class 'dict'>
k_model type: <class 'int'>
n_trials type: <class 'int'>
log_likelihood type: <class 'numpy.float64'>
AIC type: <class 'numpy.float64'>
BIC type: <class 'numpy.float64'>
LPT type: <class 'numpy.float64'>
LPT_AIC type: <class 'numpy.float64'>
LPT_BIC type: <cla